Data Source:

The core data source for this project is a list of deaths in National Park units as reported by the US National Park Service in a file downloaded from this page: https://www.nps.gov/aboutus/mortality-data.htm. I'll start with required imports and loading that dataset to a pandas dataframe.

In [3]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import geopandas as gpd

import requests, os, time, lxml

from dotenv import load_dotenv

from bs4 import BeautifulSoup

In [4]:
API_KEY = os.getenv("NPS_API_KEY")

print(API_KEY is not None)

True


In [27]:
df = pd.ExcelFile(r"../data/NPS-Mortality-Data-CY2007-to-CY2024-Released-August-2024.xlsx")
df = df.parse(sheet_name="CY2007-Present Q2")
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity
0,2007-01-01,Glen Canyon National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,65+,Not Reported
1,2007-01-22,Golden Gate National Recreation Area,Drowning,Drowning,Unintentional,Fatal injury,Male,Not Reported,Vessel Related
2,2007-01-22,Golden Gate National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,Not Reported,Vessel Related
3,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,15-24,Driving
4,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,45-54,Driving


I'm going to create a function that will clean the columns that have strings of text.

In [44]:
def clean(col):
    col = col.str.strip()
    col = col.str.lower()
    col = col.str.replace(r"\s+", " ", regex=True)
    col = col.replace("", pd.NA)

    return col

In [29]:
df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death',
       'Cause of Death Group \n(Used in the NPS Mortality Dashboard) ',
       'Intent', 'Outcome', 'Sex', 'Age Range', 'Activity'],
      dtype='object')

In [45]:
cols = ["Park Name", 
        "Intent", 
        "Outcome", 
        "Cause of Death", 
        "Cause of Death Group \n(Used in the NPS Mortality Dashboard) ", 
        "Sex", 
        "Activity"]

df[cols] = df[cols].apply(clean)

In [46]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,fatal injury,male,65+,not reported
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,fatal injury,male,Not Reported,vessel related
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,fatal injury,male,Not Reported,vessel related
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,fatal injury,female,15-24,driving
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,fatal injury,female,45-54,driving


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4635 entries, 0 to 4634
Data columns (total 9 columns):
 #   Column                                                        Non-Null Count  Dtype         
---  ------                                                        --------------  -----         
 0   Incident Date                                                 4635 non-null   datetime64[ns]
 1   Park Name                                                     4635 non-null   object        
 2   Cause of Death                                                4635 non-null   object        
 3   Cause of Death Group 
(Used in the NPS Mortality Dashboard)   4635 non-null   object        
 4   Intent                                                        4635 non-null   object        
 5   Outcome                                                       4635 non-null   object        
 6   Sex                                                           4635 non-null   object        
 7   Age Ran

In [56]:
df.isna().sum()

Incident Date                                                     0
Park Name                                                         0
Cause of Death                                                    0
Cause of Death Group \n(Used in the NPS Mortality Dashboard)      0
Intent                                                            0
Outcome                                                           0
Sex                                                               0
Age Range                                                        12
Activity                                                          0
dtype: int64

I'm going to look more closely at the "Cause of death" and "Cause of Death Group \n..." columns to see if this data is redundant, or if I need to use both in my analysis.

Check if all values in all rows and columns are the same.

In [50]:
(df["Cause of Death"] == df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]).all()

np.False_

Since the are not duplicates, I'm going to build a mask to count how many mismatches are in the data.

In [51]:
mask_mismatch = df["Cause of Death"] != df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]
print("Total rows:", len(df))
print("Total mismatches:", mask_mismatch.sum())

Total rows: 4635
Total mismatches: 494


In [52]:
df.loc[mask_mismatch, ["Cause of Death", "Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]].head()

,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard)
8,avalanche,environmental
9,vessel incident,other transportation
46,vessel incident,other transportation
48,bicycle only crash,other transportation
56,poisoning - drugs,poisoning


First, I'll focus on "Cause of Death" fields.

In [53]:
df.groupby("Cause of Death").size()

Cause of Death
aircraft incident                          77
altitude                                    1
asphyxiation                                7
avalanche                                  41
bicycle only crash                         23
collapsing earth/sand                       2
drowning                                  864
electrocution                               4
fall                                      505
falling ice                                 2
falling tree/branch                        21
fire/burn                                   5
firearm                                     5
flash flood                                14
homicide                                   51
horseback riding incident                   4
hyperthermia                               83
hypothermia                                46
legal intervention                          2
lightning strike                           10
medical - during physical activity        318
medical - not durin

In [54]:
df["Cause of Death"].nunique()

42

In [55]:
print(sorted(df["Cause of Death"].unique()))

['aircraft incident', 'altitude', 'asphyxiation', 'avalanche', 'bicycle only crash', 'collapsing earth/sand', 'drowning', 'electrocution', 'fall', 'falling ice', 'falling tree/branch', 'fire/burn', 'firearm', 'flash flood', 'homicide', 'horseback riding incident', 'hyperthermia', 'hypothermia', 'legal intervention', 'lightning strike', 'medical - during physical activity', 'medical - not during physical activity', 'medical - unknown', 'motor vehicle crash', 'other', 'poisoning - alcohol', 'poisoning - carbon monoxide', 'poisoning - drugs', 'poisoning - other', 'rockfall', 'skateboard incident', 'skiing incident', 'snowboard incident', 'snowmobile incident', 'struck by/against', 'suicide', 'train crash', 'trolly crash', 'undetermined', 'vessel incident', 'water diving incident', 'widlife incident']


Now I'll look at age range fields.

In [57]:
print(df["Age Range"].unique())

['65+' 'Not Reported' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14' nan
 '45 - 54' '35 - 44' 'Unintentional' '0 - 14']


In [68]:
unknown_age = {"Not Reported", "Unintentional", "Unknown", None, np.nan}

df["age_range_clean"] = df["Age Range"].replace(list(unknown_age), "Unknown")

print(df["age_range_clean"].value_counts())

age_range_clean
65+        827
55-64      676
Unknown    643
45-54      625
25-34      616
15-24      594
35-44      541
0-14       109
45 - 54      2
35 - 44      1
0 - 14       1
Name: count, dtype: int64


In [69]:
df["age_range_clean"] = (
    df["age_range_clean"]
    .str.replace(r"\s*-\s*", "-", regex=True)  # normalize dashes
    .str.strip()
)

In [70]:
print(df["age_range_clean"].value_counts())

age_range_clean
65+        827
55-64      676
Unknown    643
45-54      627
25-34      616
15-24      594
35-44      542
0-14       110
Name: count, dtype: int64


In [71]:
df["Age Range"] = df["age_range_clean"]

df = df.drop(columns=["age_range_clean"])

In [76]:
print(df["Age Range"].unique())

['65+' 'Unknown' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14']


In [79]:
ranges = df["Age Range"].str.extract(r"(?P<min>\d+)-(?P<max>\d+)")
df["Age Range Min"] = ranges["min"].astype(float)
df["Age Range Max"] = ranges["max"].astype(float)

mask_plus = df["Age Range"].str.endswith("+", na=False)
df.loc[mask_plus, "Age Range Min"] = df.loc[mask_plus, "Age Range"].str[:-1].astype(float)
df.loc[mask_plus, "Age Range Max"] = np.nan 

In [84]:
df.dtypes

Incident Date                                                    datetime64[ns]
Park Name                                                                object
Cause of Death                                                           object
Cause of Death Group \n(Used in the NPS Mortality Dashboard)             object
Intent                                                                   object
Outcome                                                                  object
Sex                                                                      object
Age Range                                                                object
Activity                                                                 object
Age Range Min                                                           float64
Age Range Max                                                           float64
dtype: object

In [86]:
print(df[["Age Range","Age Range Min","Age Range Max"]].drop_duplicates())

   Age Range  Age Range Min  Age Range Max
0        65+           65.0            NaN
1    Unknown            NaN            NaN
3      15-24           15.0           24.0
4      45-54           45.0           54.0
7      25-34           25.0           34.0
10     55-64           55.0           64.0
11     35-44           35.0           44.0
13      0-14            0.0           14.0
